# Part 1 — OLS, Hat Matrix, Ridge Regression & VIF

**Group 12 — Data Fitting and OLS Methods**

Notebook này kiểm chứng 4 hàm cài đặt:
- `ols_fit` — OLS qua Economic SVD (`utils/svd_solver.py → utils/decomposition.py`)
- `hat_matrix` — Ma trận chiếu H = U_r Uᵣᵀ qua SVD
- `ridge_fit` — Ridge Regression qua Gauss-Jordan inverse
- `vif` — Variance Inflation Factor qua hồi quy phụ OLS

In [ ]:
import sys, os
# Them thu muc goc du an vao sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from part1.ols_implementation import ols_fit, hat_matrix, vif
from part1.ridge_lasso import ridge_fit
from utils.decomposition import svd_decomp   # ham SVD

rng = np.random.default_rng(42)
print('Import thanh cong!')
print('svd_decomp den tu utils/decomposition.py')

## Cài đặt và Kiểm chứng các phép đo Thống kê & Cross-Validation cho OLS
Phần này thực hiện chẩn đoán mô hình, suy diễn hệ số ($t$-test, $F$-test), phân tích phần dư và K-fold CV. 
Các công thức toán học chi tiết (như ma trận hiệp phương sai, phân rã $TSS = ESS + RSS$) được chứng minh trong file Báo cáo (PDF).

### 1. Suy diễn hệ số ($t$-test) và Thống kê tổng quát ($F$-test, $R^2$)
Kiểm tra tính chính xác của hàm `model_metrics` và `coef_inference` so với thư viện `statsmodels`.

**1. Đánh giá tổng thể mô hình:**
* **Vector phần dư (Residuals):** $e = y - \hat{y}$
* **Tổng bình phương phần dư (RSS):** $RSS = \sum_{i=1}^n (y_i - \hat{y}_i)^2 = e^T e$
* **Vector trung tâm (Centered Target):** $y_c = y - \bar{y}$
* **Tổng bình phương toàn phần (TSS):** $TSS = \sum_{i=1}^n (y_i - \bar{y})^2 = y_c^T y_c$
* **Hệ số xác định ($R^2$):** $R^2 = 1 - \frac{RSS}{TSS}$
* **$R^2$ hiệu chỉnh:** $R^2_{adj} = 1 - \left( \frac{RSS / (n - p - 1)}{TSS / (n - 1)} \right)$
* **Kiểm định $F$:** $F = \frac{(TSS - RSS) / p}{RSS / (n - p - 1)}$

**2. Suy diễn hệ số (Inference):**
* **Ma trận hiệp phương sai:** $\text{Var}(\hat{\beta}) = \hat{\sigma}^2 (X^T X)^{-1}$ với $\hat{\sigma}^2 = \frac{RSS}{n - p - 1}$
* **Sai số chuẩn ($SE$):** $SE(\hat{\beta}_j) = \sqrt{\text{Var}(\hat{\beta})_{jj}}$
* **Kiểm định $t$:** $t_j = \frac{\hat{\beta}_j}{SE(\hat{\beta}_j)}$
* **Khoảng tin cậy 95%:** $CI = \hat{\beta}_j \pm t_{\alpha/2, n-p-1} \times SE(\hat{\beta}_j)$

---
## 1. OLS Fit — `ols_fit(X, y)`

### Lý thuyết
Ước lượng OLS tối thiểu hóa $\|y - X\beta\|^2$, cho nghiệm:
$$\hat{\beta} = (X^\top X)^{-1} X^\top y$$

### Cài đặt
Dùng Economic SVD: $X = U\Sigma V^\top$
$$\hat{\beta} = V\Sigma^{-1}U^\top y$$

Luồng gọi: `ols_fit` → `svd_solve` (utils/svd_solver.py) → `economic_svd` → **`svd_decomp`** (utils/decomposition.py)

In [ ]:
# Sinh du lieu gia lap
n, p = 100, 3
X_raw = rng.standard_normal((n, p))
X = np.hstack([np.ones((n, 1)), X_raw])   # them cot he so chan
true_beta = np.array([2.0, 1.5, -1.0, 0.5])
y = X @ true_beta + rng.standard_normal(n) * 1.5

# OLS
beta_hat = ols_fit(X, y)

# Kiem chung bang numpy.linalg.lstsq
beta_ref = np.linalg.lstsq(X, y, rcond=None)[0]

print('=== OLS Fit Verification ===')
print(f'beta_hat (our SVD) : {beta_hat}')
print(f'beta_ref (lstsq)   : {beta_ref}')
print(f'Max absolute error : {np.max(np.abs(beta_hat - beta_ref)):.2e}')
assert np.allclose(beta_hat, beta_ref, atol=1e-8), 'FAIL: ols_fit khong khop voi lstsq!'
print('PASS: ols_fit khop voi numpy.linalg.lstsq den do chinh xac may!')

---
## 2. Hat Matrix — `hat_matrix(X)`

### Lý thuyết
$$H = X(X^\top X)^{-1}X^\top, \quad \hat{y} = Hy$$

### Cài đặt qua SVD
$$H = U_r U_r^\top \quad \text{(với } U_r \text{ là các cột của } U \text{ ứng với } \sigma_i > \varepsilon\text{)}$$

Tính chất cần kiểm chứng: lũy đẳng ($H^2=H$), đối xứng ($H^\top=H$), $\text{tr}(H)=p+1$.

In [ ]:
H = hat_matrix(X)

H2_err  = np.max(np.abs(H @ H - H))
sym_err = np.max(np.abs(H - H.T))
tr_val  = np.trace(H)

print('=== Hat Matrix Properties ===')
print(f'Shape                    : {H.shape}')
print(f'Idempotent |H^2-H| max   : {H2_err:.2e}  ({"PASS" if H2_err < 1e-8 else "FAIL"})')
print(f'Symmetric  |H-H^T| max   : {sym_err:.2e}  ({"PASS" if sym_err < 1e-8 else "FAIL"})')
print(f'tr(H) = {tr_val:.6f}  (expected {p+1})  ({"PASS" if abs(tr_val-(p+1)) < 1e-8 else "FAIL"})')

# Kiem chung bang tinh truc tiep
H_ref = X @ np.linalg.solve(X.T @ X, X.T)
print(f'Max diff vs direct H_ref : {np.max(np.abs(H - H_ref)):.2e}')

### 2. Phân tích phần dư & Đánh giá giả định Gauss-Markov

Để kiểm chứng toàn diện, ta vẽ 4 biểu đồ chẩn đoán chuẩn mực (tương đương `plot(lm)` trong R) trên 2 kịch bản dữ liệu giả lập:

**Công thức Toán học cài đặt:**
* **Giá trị đòn bẩy (Leverage):** $h_{ii}$ là các phần tử trên đường chéo của Hat Matrix $H = X(X^T X)^{-1}X^T$.
* **Phần dư chuẩn hóa (Standardized Residuals):** $r_i = \frac{e_i}{\hat{\sigma} \sqrt{1 - h_{ii}}}$
* **Q-Q Plot:** Sử dụng phép xấp xỉ Tukey Lambda để tính phân vị lý thuyết, không phụ thuộc vào thư viện `scipy`.

**Kịch bản 1: Dữ liệu chuẩn (Phương sai không đổi)**
* *Nhận xét Biểu đồ:* * **Residuals vs Fitted:** Các điểm phân tán ngẫu nhiên quanh trục 0 $\rightarrow$ Thỏa mãn tính tuyến tính.
    * **Normal Q-Q:** Bám sát đường chéo đỏ $\rightarrow$ Phần dư tuân theo phân phối chuẩn.
    * **Scale-Location:** Đường xu hướng nằm ngang $\rightarrow$ Thỏa mãn Homoscedasticity (Phương sai đồng đều).
    * **Residuals vs Leverage:** Không có điểm nào nằm tách biệt ở góc phải (Leverage cao) với phần dư lớn $\rightarrow$ Không có điểm ảnh hưởng xấu (Influential Outliers).

**Kịch bản 2: Dữ liệu lỗi (Phương sai thay đổi - Heteroscedasticity)**
* *Nhận xét Biểu đồ:* * **Scale-Location:** Biểu đồ có hình cái phễu (phân tán rộng dần về bên phải) $\rightarrow$ Vi phạm giả định phương sai không đổi. Mô hình OLS tuy không chệch nhưng mất đi tính hiệu quả tối ưu (không còn là BLUE).

---
## 3. Ridge Regression — `ridge_fit(X, y, lam)`

### Lý thuyết
$$\hat{\beta}_{\text{ridge}} = (X^\top X + \lambda I)^{-1} X^\top y$$

### Cài đặt
Dùng Gauss-Jordan inverse (`utils/inverse.py`). Ma trận $(X^\top X + \lambda I)$ luôn khả nghịch với $\lambda > 0$.

In [ ]:
# Du lieu co da cong tuyen
X_mc = np.hstack([np.ones((n,1)), X_raw[:,0:1], X_raw[:,1:2],
                  0.95*X_raw[:,0:1] + 0.05*rng.standard_normal((n,1))])
y_mc = X_mc @ np.array([1,2,-1,0.5]) + rng.standard_normal(n)*1.5

lambdas = np.logspace(-3, 2, 50)
betas_ridge = [ridge_fit(X_mc, y_mc, lam) for lam in lambdas]

# Ridge Trace
plt.figure(figsize=(8, 4))
for j in range(X_mc.shape[1]):
    plt.plot(np.log10(lambdas), [b[j] for b in betas_ridge], label=f'beta[{j}]')
plt.axhline(0, color='k', lw=0.5)
plt.xlabel('log10(lambda)')
plt.ylabel('Coefficient value')
plt.title('Ridge Trace (he so tieu dan ve 0 khi lambda tang)')
plt.legend()
plt.tight_layout()
plt.show()

# Kiem chung tai mot lambda cu the
lam_test = 1.0
b_ours = ridge_fit(X_mc, y_mc, lam_test)
b_ref  = np.linalg.solve(X_mc.T@X_mc + lam_test*np.eye(X_mc.shape[1]), X_mc.T@y_mc)
print(f'=== Ridge Fit Verification (lambda={lam_test}) ===')
print(f'Max error vs numpy.linalg.solve: {np.max(np.abs(b_ours - b_ref)):.2e}')
assert np.allclose(b_ours, b_ref, atol=1e-8), 'FAIL'
print('PASS')

### 3. K-Fold Cross-Validation: Đánh giá mô hình & Phát hiện Overfitting

**1. Phân chia dữ liệu (Data Splitting):**
* Tập dữ liệu gốc có kích thước $n$ được chia thành $k$ phần (folds) rời rạc có kích thước xấp xỉ nhau: $S_1, S_2, \dots, S_k$.

**2. Vòng lặp K-Fold (Với mỗi fold $i$ từ $1$ đến $k$):**
* **Phân tách tập:** Chọn phần $S_i$ làm tập kiểm định (Validation set) chứa $m$ mẫu. Các phần còn lại $S \setminus S_i$ tạo thành tập huấn luyện (Train set).
* **Huấn luyện mô hình:** Ước lượng vector hệ số $\hat{\beta}^{(i)}$ trên tập Train bằng phương pháp OLS.
* **Dự đoán:** Tính vector giá trị dự đoán trên tập Validation: 
  $$\hat{y}_{val}^{(i)} = X_{val}^{(i)} \hat{\beta}^{(i)}$$
* **Tính lỗi của fold (Mean Squared Error - MSE):**
  $$MSE_i = \frac{1}{m} \sum_{j=1}^{m} \left( y_{val, j}^{(i)} - \hat{y}_{val, j}^{(i)} \right)^2$$

**3. Điểm đánh giá tổng hợp (CV Score):**
* Lỗi K-Fold CV là trung bình cộng của lỗi trên tất cả $k$ folds:
  $$CV\_Score = \frac{1}{k} \sum_{i=1}^k MSE_i$$

---
## 4. VIF — `vif(X)`

### Lý thuyết
$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$

$R_j^2$ là hệ số xác định khi hồi quy $x_j$ lên các biến còn lại. Cài đặt: gọi `ols_fit` cho từng biến, tính $R^2$ bằng vòng lặp.

In [ ]:
print('=== Scenario 1: Bien doc lap (VIF ~ 1) ===')
X_ind = rng.standard_normal((n, 3))
vif_df = vif(X_ind)
print(vif_df.to_string(index=False))
assert vif_df['VIF_Score'].max() < 5, 'FAIL: VIF qua cao cho du lieu doc lap'
print('PASS: VIF gan 1 khi cac bien doc lap\n')

print('=== Scenario 2: Da cong tuyen manh (VIF >> 10) ===')
X_col = np.column_stack([
    X_ind[:,0],
    X_ind[:,1],
    X_ind[:,0]*0.99 + 0.01*rng.standard_normal(n)
])
vif_df2 = vif(X_col)
print(vif_df2.to_string(index=False))
assert vif_df2['VIF_Score'].max() > 10, 'FAIL: VIF phai cao khi co da cong tuyen'
print('PASS: VIF rat cao khi co bien gan tuong quan tuyen tinh')

### 4. Mô phỏng Monte Carlo định lý Gauss-Markov

---
## Tổng kết

| Hàm | Phương pháp | Kiểm chứng | Kết quả |
|---|---|---|---|
| `ols_fit` | Economic SVD (`svd_decomp`) | vs `numpy.linalg.lstsq` | max err < 1e-8 |
| `hat_matrix` | H = UᵣUᵣᵀ từ SVD | H²=H, H=Hᵀ, tr(H)=p+1 | max err < 1e-16 |
| `ridge_fit` | Gauss-Jordan inverse | vs `numpy.linalg.solve` | max err < 1e-8 |
| `vif` | Hồi quy phụ OLS + R² | VIF~1 (độc lập), VIF>>10 (cộng tuyến) | PASS |

**Tất cả SVD đều dùng `svd_decomp` từ `utils/decomposition.py`. `np.linalg.svd` không được sử dụng trong bất kỳ hàm nào.**